In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Paper reconstruction supplement worker

Thin fixed entry. Scientific semantics and recovery live in the tested runner; this notebook only mounts Drive, checks out the bound producer, and launches one stable JOB_ID.


In [ ]:
import json, os, pathlib, subprocess, sys
from google.colab import userdata

REPO='https://github.com/RICHAAARC/CEG-WM.git'
EXPECTED_EXACT='__PAPER_FORMAL_PRODUCER_EXACT__'
JOB_ID='paper-main-reconstruction-v1'
MAIN_JOB_ID='paper-main-v1'
checkout=pathlib.Path('/content/cegwm-paper-reconstruction')
runtime_root=pathlib.Path('/content/cegwm-paper-runtime/reconstruction-detection')
drive_root=pathlib.Path('/content/drive/MyDrive/CEG-WM/PaperFormal-V1')
subprocess.run(['git','clone',REPO,str(checkout)],check=True)
subprocess.run(['git','-C',str(checkout),'checkout','--detach',EXPECTED_EXACT],check=True)
head=subprocess.run(['git','-C',str(checkout),'rev-parse','HEAD'],check=True,capture_output=True,text=True).stdout.strip()
dirty=subprocess.run(['git','-C',str(checkout),'status','--porcelain'],check=True,capture_output=True,text=True).stdout.strip()
assert head==EXPECTED_EXACT and not dirty
subprocess.run([sys.executable,'-m','pip','install','diffusers<0.40','transformers','accelerate'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-e',str(checkout)],check=True)
child_env=dict(os.environ)
child_env['HF_TOKEN']=userdata.get('HF_TOKEN') or ''
child_env['CEG_WM_ROOT_KEY']=userdata.get('CEG_WM_ROOT_KEY') or ''
assert child_env['HF_TOKEN'] and child_env['CEG_WM_ROOT_KEY']
command=[sys.executable,'-m','experiments.run_paper_reconstruction_worker','--job-id',JOB_ID,'--main-job-id',MAIN_JOB_ID,'--expected-exact',EXPECTED_EXACT,'--drive-root',str(drive_root),'--runtime-root',str(runtime_root)]
subprocess.run(command,cwd=checkout,env=child_env,check=True)
final_path=drive_root/'reconstruction'/JOB_ID/'reconstruction_final.json'
public=json.loads(final_path.read_text(encoding='utf-8'))
print({'method_id':public['method_id'],'status':public['status'],'fpr_resolution':public['fpr_resolution']})
